# GBM-Neuro Cocultures — Machine Learning Analysis
## Student Notebook

Welcome! In this notebook you will use machine learning to explore a real neuroscience dataset.

**You will not write code yourself.** Each section contains a prompt that you copy and paste into an AI assistant (e.g. Claude or ChatGPT). The AI will generate the code, which you then paste into the empty code cell below and run.

---
### How to use this notebook
1. Read the explanation in each section carefully — understanding *what* and *why* matters more than the code itself.
2. Copy the prompt from the grey box and paste it into your AI assistant.
3. Paste the code it gives you into the empty code cell below. Try to understand the code.
4. Run the cell and study the output. Does it make sense? Is it in line with your expectations?
5. If something goes wrong, paste the error message back into the AI and ask it to fix it.
6. Answer the reflection questions — these are the most important part.

---
### The dataset
You are working with electrophysiology recordings from **neural organoids** — tiny brain-like structures grown in a dish from human stem cells. Four conditions are compared:
- **Patient1** and **Patient2**: Neurons cocultured with GBM (glioblastoma) patient cells
- **Astro**: Neurons co-cultured with astrocytes
- **Control**: Only neurons

Each row in the dataset is one recording from one network at one time point. The columns describe how the neurons behave electrically — how fast they fire, how they burst, and how synchronised they are.

**DIV** = Days In Vitro — how many days the cultures have been growing in the dish. At DIV 28, the cancer cells and astrocytes are added.

---
### Our goal
Can we tell the four conditions apart just from their electrical activity? And which electrical features are most responsible for the differences?

## 1. Imports & global settings

We load all the libraries we need up front. A **library** is a collection of pre-written functions so we don't have to code everything from scratch:
- **pandas / numpy**: handle data tables and numerical operations
- **matplotlib / seaborn**: create all plots
- **sklearn**: machine learning tools — PCA, Random Forest, cross-validation, evaluation metrics

We also fix `RANDOM_STATE = 42`. Many algorithms involve internal randomness (e.g. how trees are built, how data is split into folds). Fixing the random seed means that every time you run this notebook you get **exactly the same results**, making the analysis fully reproducible.

In [ ]:
# !pip install scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.inspection import permutation_importance

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# One consistent color per condition — the same color will be used in every
# single plot so you never have to re-learn which color represents which group
PALETTE = {
    "Patient1": "#E63946",
    "Patient2": "#457B9D",
    "Astro":    "#2A9D8F",
    "Control":  "#E9C46A",
}
CONDITIONS = list(PALETTE.keys())

print("All imports successful.")

---
## Section 2 — Load & inspect the data

Before any analysis, we always look at the data first:
- How many recordings per condition? Are the groups balanced in size?
- What is the DIV range?
- Are there any missing values?

Understanding the data structure prevents mistakes later.

> **Prompt:**
> ```
> Write a Python cell that:
> 1. Loads a CSV from this path into a dataframe called df:
>    C:\Users\Giuli\Documents\Nigeria\csvs\MERGED_data_cleaned.csv
>    Use Path from pathlib and a raw string.
> 2. Prints the shape of df (number of rows and columns).
> 3. Prints the value counts of the "Patient_Status" column.
> 4. Prints the min and max of the "DIV" column.
> 5. Prints the count of missing values per column (only columns with > 0 missing);
>    if none are missing prints "None!".
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- How many recordings does each condition have? Is the dataset balanced?
- Are all DIV bins represented equally for every condition?
- Are there any missing values? Why might that matter?

---
## Section 3 — Prepare the feature matrix

### What is a feature matrix?
Machine learning models work with rectangular tables of numbers:
- Each **row** = one observation (here: one network recording at one time point)
- Each **column** = one **feature** (a number measuring some aspect of neural activity)

We separate **metadata** (who/when a recording is — e.g. `filename`, `DIV`, `Patient_Status`) from **features** (the actual measurements — e.g. `FR`, `BR`, `Mean_Corr_Coeff`). Only the features go into the model.

### Why standardize?
Different features have very different scales: firing rate (FR) ranges 0–70 Hz, while correlation coefficient ranges 0–1. Without standardization, the model would pay far more attention to FR simply because its numbers are bigger — not because it is more informative. Standardization transforms every feature to **mean = 0, standard deviation = 1**, so each feature starts on equal footing.

> **Prompt:**
> ```
> I have a pandas dataframe df. Write a Python cell that:
> 1. Defines META_COLS as a list containing:
>    ["filename", "chip_ID", "NW_ID", "DIV", "DIV_bin", "Patient_Status", "Source_Folder"]
>    then filters to only keep columns that actually exist in df.
> 2. Defines FEATURE_COLS as all columns in df that are NOT in META_COLS.
> 3. Prints how many features there are and lists their names.
> 4. Creates X_raw = df[FEATURE_COLS].values  (raw feature matrix, numpy array)
>    and y = df["Patient_Status"].values  (condition labels)
> 5. Uses StandardScaler from sklearn to standardize X_raw and stores the result in X.
> 6. Uses LabelEncoder to encode y to integers, storing the result in y_enc and the encoder in le.
> 7. Prints the label encoding mapping.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- Which columns are excluded as metadata and which are included as features?
- Why do we standardize the features before any analysis?

---
# PART A — Unsupervised Exploration: PCA

In this part the algorithm **never sees the condition labels**. We ask: if you look at the numbers alone, does the data naturally organise itself by condition?

## What is PCA?

We have many features, so each recording lives in a multi-dimensional space — impossible to visualise. **Principal Component Analysis (PCA)** compresses this into 2 dimensions while preserving as much variation as possible.

Think of it like finding the best camera angle for a complex 3D sculpture: the right angle shows you the most structure in a flat 2D photograph.

- **PC1** = the direction that captures the most variation across all recordings
- **PC2** = the next-best direction, perpendicular to PC1
- The **scree plot** shows how much variance each PC captures — look for an elbow
- The **loadings** tell you which original features drive each PC

### Trajectory plots
We fit PCA once on all data, then plot one panel per DIV bin using **identical axis limits** across all panels. Because the axes are fixed, you can directly compare positions across time. The star (★) marks the **centroid** (mean position) of each condition — watching the stars move reveals the **developmental trajectory** of each condition.

---
## Section 4a — Global PCA: scree plot + 2D scatter

> **Prompt:**
> ```
> I have a standardized feature matrix X (numpy array) and condition labels y (string array).
> I also have a color palette PALETTE = {"Patient1": "#E63946", "Patient2": "#457B9D",
> "Astro": "#2A9D8F", "Control": "#E9C46A"}.
>
> Write a Python cell that:
> 1. Fits a PCA (random_state=42) on X and stores the result in X_pca.
>    Stores the PCA object as pca_global.
>    Computes var_exp = explained variance ratio * 100 (% per component).
>    Computes cumvar = cumulative sum of var_exp.
> 2. Makes a figure with 3 subplots side by side (figsize=(16,4)):
>    - Left: bar chart of var_exp for the first 10 components
>      (xlabel="Principal Component", ylabel="Variance explained (%)",
>       title="Scree Plot\n(how much information each PC captures)")
>    - Middle: line plot of cumvar vs number of components, with a red dashed
>      horizontal line at 80%, labeled "80% threshold"
>      (xlabel="Number of Components", ylabel="Cumulative variance (%)",
>       title="Cumulative Explained Variance")
>    - Right: scatter plot of PC1 vs PC2 for all recordings, colored by condition
>      using PALETTE (alpha=0.6, s=30), with a legend
>      (xlabel = "PC1 (X.X% variance)", ylabel = "PC2 (X.X% variance)",
>       title="PCA — all time points\n(no label info used to make this plot)")
> 3. Saves the figure as "pca_global.png" (dpi=150) and shows it.
> 4. Prints the number of PCs needed to explain 80% of the variance
>    (use np.searchsorted(cumvar, 80) + 1).
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- How many PCs are needed to capture 80% of the information? What does that tell you about the complexity of the data?
- In the 2D scatter plot, do the four conditions cluster separately or overlap? What does that suggest?

---
## Section 4b — PCA loadings

The **loadings** tell us which original features contribute most to each PC axis. A large positive loading means that feature pushes recordings in the positive direction on that axis; a large negative loading pushes them in the negative direction. Features near zero barely contribute.

> **Prompt:**
> ```
> I have a fitted PCA object pca_global and a list FEATURE_COLS of feature names.
> Write a Python cell that:
> 1. Creates a DataFrame called loadings with pca_global.components_[:2].T as values,
>    index=FEATURE_COLS, and columns=["PC1", "PC2"].
> 2. Makes a figure with 2 subplots side by side (figsize=(14,5)).
>    For each of PC1 and PC2:
>    - Sort the loadings for that component in ascending order.
>    - Plot a horizontal bar chart (kind="barh") with bars colored #E63946 if positive,
>      #457B9D if negative.
>    - Add a vertical black line at x=0.
>    - Title: "{PC} Loadings\n(which features define this axis; red=positive, blue=negative)"
>    - xlabel: "Loading weight"
> 3. Saves as "pca_loadings.png" (dpi=150) and shows it.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- Which features have the strongest positive loading on PC1? What do those features measure biologically?
- Which features define PC2? Do these make biological sense given what you know about the conditions?

---
## Section 4c — PCA trajectory per DIV bin

Now we show one PCA panel per DIV bin, all with the **same fixed axis limits** so positions are directly comparable across time. The star (★) marks the centroid (mean position) of each condition. Watch how the centroids move from panel to panel — that is the developmental trajectory.

> **Prompt:**
> ```
> I have:
> - X_pca: numpy array of PCA coordinates (n_samples x n_components)
> - pca_global: fitted PCA object with explained_variance_ratio_
> - y: numpy array of condition labels (strings)
> - df: dataframe with a column "DIV_bin" (or "DIV" if "DIV_bin" is absent)
> - PALETTE: dict mapping condition names to hex colors
> - mpatches: imported as matplotlib.patches
>
> Write a Python cell that:
> 1. Determines div_col = "DIV_bin" if it exists in df, else "DIV".
>    Computes div_bins = sorted unique values of df[div_col].
> 2. Builds a DataFrame pca_df with columns PC1, PC2, Patient_Status, and div_col.
> 3. Computes centroids = mean PC1 and PC2 for each (Patient_Status, div_col) combination.
> 4. Sets ncols=4, nrows=ceil(len(div_bins)/4).
>    Sets xlim and ylim from the global min/max of X_pca[:,0] and X_pca[:,1] (+/- 0.5 padding).
> 5. Creates a figure with nrows x ncols subplots, figsize=(5*ncols, 4*nrows), axes flattened.
> 6. For each DIV bin:
>    - Plot all recordings as small semi-transparent dots colored by PALETTE (alpha=0.45, s=22).
>    - Plot the centroid of each condition as a large star (marker="*", s=220,
>      edgecolors="black", linewidths=0.5, zorder=5).
>    - Set title to "DIV {div}  (n={n})" where n is the number of recordings in that bin.
>    - Set xlim, ylim from step 4. Add xlabel="PC1", ylabel="PC2".
>    - Add faint dashed horizontal and vertical lines at 0.
> 7. Hides any unused subplots.
> 8. Adds a shared legend using mpatches.Patch, placed at lower right, title="Condition".
> 9. Adds a suptitle: "PCA per DIV bin — same global axes across all panels\n★ = condition centroid"
>    (fontsize=14, y=1.01).
> 10. Saves as "pca_per_div.png" (dpi=150, bbox_inches="tight") and shows it.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- At the earliest DIV bins, are the condition centroids (★) close together or far apart?
- At which DIV bin do the centroids start to visibly separate?
- Do the conditions follow similar trajectories (parallel paths) or diverge strongly? What might that suggest biologically?

---
# PART B — Supervised Classification: Random Forest

## What is a Random Forest?

A **decision tree** is a flowchart of yes/no questions:
> "Is FR > 5 Hz?" → Yes → "Is IBIcv > 0.4?" → No → Predict: *Patient1*

A single tree can **overfit** — memorise the training data and fail on new recordings. A **Random Forest** builds hundreds of trees, each trained on a random subset of samples and features. The final prediction is a **majority vote**. This makes the model much more reliable and less prone to overfitting.

## Why cross-validation — and what is it?

If we tested the model on the data it trained on, it would already know the answers — like testing students with the exact questions they studied. We need to test on **data the model has never seen**.

**Stratified 5-fold cross-validation:**
1. Split the data into 5 equal-sized groups (folds), keeping class proportions balanced in each
2. Train on 4 folds → test on the 1 held-out fold
3. Repeat 5 times, rotating which fold is held out
4. Every recording is tested exactly once; report the average and spread across folds

## How to read the metrics
- **Accuracy**: percentage of recordings correctly classified
- **Balanced accuracy**: accuracy averaged per class — fairer when some conditions have more recordings than others
- **F1-macro**: harmonic mean of precision and recall, averaged equally across all four conditions
- **Random chance**: with 4 conditions, a model that just guesses randomly = 25%

---
## Section 5 — Train & evaluate the Random Forest

> **Prompt:**
> ```
> I have:
> - X: standardized feature matrix (numpy array, shape n_samples x n_features)
> - y_enc: integer-encoded condition labels (numpy array)
> - RANDOM_STATE = 42
>
> Write a Python cell that:
> 1. Creates a RandomForestClassifier with:
>    n_estimators=500, max_features="sqrt", class_weight="balanced",
>    random_state=RANDOM_STATE, n_jobs=-1.
>    Store it as rf.
>    Add a comment after each parameter explaining what it does.
> 2. Creates a StratifiedKFold with n_splits=5, shuffle=True, random_state=RANDOM_STATE.
>    Store it as cv.
> 3. Runs cross_validate with scoring=["accuracy", "balanced_accuracy", "f1_macro"]
>    and return_train_score=True. Prints "Running 5-fold cross-validation..."
>    before the call.
> 4. Creates a results DataFrame with columns:
>    Fold (1-5), Train acc., Test acc., Test bal. acc., Test F1-macro.
>    Prints it without the index.
> 5. Prints mean +/- std for test accuracy, balanced accuracy, and F1-macro.
> 6. Prints a reminder that random chance = 0.25 and that a large Train/Test gap = overfitting.
> ```

In [ ]:
# Paste the code from your AI here

> **Prompt:**
> ```
> I have cv_results (output of sklearn cross_validate) with keys
> "test_accuracy", "test_balanced_accuracy", "test_f1_macro".
>
> Write a Python cell that:
> 1. Creates a figure (figsize=(7,4)) with one axis.
> 2. For i, (label, key) in enumerate of a dict:
>    {"Accuracy": "test_accuracy",
>     "Balanced\nAccuracy": "test_balanced_accuracy",
>     "F1 Macro": "test_f1_macro"}
>    plots a boxplot of cv_results[key] at position i, widths=0.4,
>    patch_artist=True, facecolor="steelblue" alpha=0.7, median line white lw=2.
> 3. Sets xticks and xticklabels from the dict keys.
> 4. Sets ylabel="Score", ylim=[0, 1.05].
> 5. Title: "5-Fold Cross-Validation Performance\nAre the conditions distinguishable?"
> 6. Adds a red dashed horizontal line at 0.25 labeled "Random chance (4 classes)".
> 7. Adds a green dotted line at 1.0 labeled "Perfect score".
> 8. Adds a legend. Saves as "cv_scores_overall.png" (dpi=150) and shows.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- How does the model's accuracy compare to random chance (25%)?
- Is there a large gap between training and test accuracy? What would that indicate?
- Which metric (accuracy, balanced accuracy, F1) is most informative here and why?

---
## Section 6 — Confusion matrix

The confusion matrix shows the **full breakdown** of the model's predictions:
- **Rows** = true condition (what the recording actually is)
- **Columns** = predicted condition (what the model said)
- **Diagonal** = correct predictions
- **Off-diagonal** = mistakes (e.g. a Patient1 recording predicted as Control)

The normalized version divides each row by the total samples in that class, so each row sums to 1.0. This makes it easy to compare per-class performance regardless of class size.

> **Prompt:**
> ```
> I have:
> - rf: a fitted RandomForestClassifier
> - X, y_enc: feature matrix and integer labels
> - cv: StratifiedKFold object
> - le: LabelEncoder with le.classes_ giving the condition names
>
> Write a Python cell that:
> 1. Uses cross_val_predict(rf, X, y_enc, cv=cv) to get out-of-fold predictions y_pred.
>    (This gives a prediction for every sample from a model that never trained on it.)
> 2. Creates a figure with 2 subplots side by side (figsize=(14,5)).
> 3. Left subplot: confusion_matrix(y_enc, y_pred), displayed with
>    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], colorbar=False, cmap="Blues")
>    Title: "Confusion Matrix — raw counts"
> 4. Right subplot: confusion_matrix normalized by true class (normalize="true"), rounded to 2 dp.
>    Same display approach. Title: "Confusion Matrix — normalized per true class\n(each row sums to 1.0)"
> 5. Saves as "confusion_matrix.png" (dpi=150) and shows.
> 6. Prints classification_report(y_enc, y_pred, target_names=le.classes_).
> 7. Prints one-line explanations of Precision, Recall, and F1.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- Which condition is most accurately classified (highest value on the diagonal in the normalized matrix)?
- Which two conditions are most often confused with each other? Can you think of a biological reason?
- Look at the classification report: which condition has the lowest F1-score?

---
# PART C — Feature Importances

## Which features does the model rely on most?

Knowing *that* the model can distinguish conditions is useful. Knowing *which features* drive those distinctions connects the machine learning result directly back to biology.

We use two complementary measures:

**1. Mean Decrease in Impurity (MDI)**
Built directly into the Random Forest. Measures how much each feature reduces prediction error across all trees. Fast to compute, but can overestimate features with many unique values.

**2. Permutation Importance** (more reliable)
After training, we randomly shuffle the values of one feature at a time — destroying any relationship between that feature and the outcome — and measure how much accuracy drops. A large drop = the model genuinely needed that feature. Near zero or negative = the model performs just as well without it. We repeat each shuffle 20 times to get stable estimates with error bars.

Features at or to the left of zero in the permutation plot are not contributing anything useful.

---
## Section 7 — MDI feature importances

> **Prompt:**
> ```
> I have:
> - rf: a RandomForestClassifier (not yet fitted on full data)
> - X, y_enc: full feature matrix and integer labels
> - FEATURE_COLS: list of feature names
>
> Write a Python cell that:
> 1. Prints "Fitting final Random Forest on all data..." then fits rf on X, y_enc.
> 2. Creates a DataFrame mdi_df with columns "feature" (FEATURE_COLS) and
>    "importance" (rf.feature_importances_), sorted ascending by importance.
> 3. Makes a horizontal bar chart (figsize=(8,6)):
>    - Bars colored #E63946 if importance > 75th percentile, else "steelblue".
>    - A vertical dashed black line at the mean importance, labeled "Mean importance".
>    - xlabel: "Mean Decrease in Impurity"
>    - title: "MDI Feature Importances\n(red = top 25% most important features)"
>    - legend included.
> 4. Saves as "feature_importance_mdi.png" (dpi=150) and shows.
> ```

In [ ]:
# Paste the code from your AI here

---
## Section 8 — Permutation importances

> **Prompt:**
> ```
> I have:
> - rf: a RandomForestClassifier already fitted on X, y_enc
> - X, y_enc: feature matrix and integer labels
> - FEATURE_COLS: list of feature names
> - RANDOM_STATE = 42
>
> Write a Python cell that:
> 1. Prints "Computing permutation importance (n_repeats=20)... may take ~1 minute."
> 2. Calls permutation_importance(rf, X, y_enc, n_repeats=20,
>    random_state=RANDOM_STATE, n_jobs=-1) and stores the result as perm.
> 3. Creates a DataFrame perm_df with columns "feature", "importance_mean"
>    (perm.importances_mean), "importance_std" (perm.importances_std),
>    sorted ascending by importance_mean.
> 4. Makes a horizontal bar chart (figsize=(8,6)) with error bars (xerr=importance_std,
>    color="steelblue", alpha=0.8, error_kw=dict(ecolor="black", capsize=3)).
> 5. Adds a red dashed vertical line at 0, labeled
>    "No effect — shuffling doesn't change accuracy".
> 6. xlabel: "Mean accuracy drop when feature is shuffled (+/- std across 20 repeats)"
> 7. title: "Permutation Importances\n(features left of zero line are not useful to the model)"
> 8. Saves as "feature_importance_permutation.png" (dpi=150) and shows.
> 9. Prints the full ranking from most to least important.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- Do the MDI and permutation importance rankings agree? If not, which features differ and why might that be?
- Which features are at or below zero in the permutation plot? What does that mean?
- Look up the top 3 features — what do they measure? Why might those properties differ between conditions?

---
## Section 9 — Top features: distributions & trajectories

Now we look at the most important features directly in the raw data, connecting the model result back to biology.

### 9a — Boxplots per condition

> **Prompt:**
> ```
> I have:
> - perm_df: DataFrame with columns "feature" and "importance_mean", sorted ascending
> - df: original dataframe with a "Patient_Status" column
> - PALETTE: dict of condition → hex color
> - CONDITIONS: list of condition names in order
>
> Write a Python cell that:
> 1. Gets top_features = the top 6 feature names from perm_df sorted descending by importance_mean.
> 2. Creates plot_df = df[top_features + ["Patient_Status"]].copy().
> 3. Makes a 2x3 grid of subplots (figsize=(14,8)).
>    For each (ax, feat) pair:
>    - seaborn boxplot: data=plot_df, x="Patient_Status", y=feat,
>      palette=PALETTE, order=CONDITIONS.
>    - title = feat, xlabel = ""
>    - rotate x-axis tick labels 15 degrees
> 4. suptitle: "Top 6 Features by Condition (all time points)", fontsize=14, y=1.01
> 5. Saves as "top_features_boxplot.png" (dpi=150) and shows.
> ```

In [ ]:
# Paste the code from your AI here

### 9b — Developmental trajectories of the top 3 features

> **Prompt:**
> ```
> I have:
> - perm_df: DataFrame with columns "feature" and "importance_mean"
> - df: original dataframe with "Patient_Status" and a "DIV_bin" column
>   (or "DIV" if "DIV_bin" doesn't exist)
> - PALETTE: dict of condition → hex color
>
> Write a Python cell that:
> 1. Gets top3 = top 3 feature names from perm_df sorted descending by importance_mean.
> 2. Sets div_col = "DIV_bin" if it exists in df, else "DIV".
> 3. Makes 1 x len(top3) subplots (figsize=(5*len(top3), 5)).
>    If len(top3)==1, wrap axes in a list.
> 4. For each (ax, feat):
>    For each condition in PALETTE:
>      - Compute median of feat grouped by div_col for that condition.
>      - Plot as a line with marker="o", linewidth=2, using the palette color.
>    - xlabel="DIV bin", ylabel=feat
>    - title=f"{feat} over time\n(median per condition per DIV bin)"
>    - legend fontsize=8, grid alpha=0.3
> 5. suptitle: "Top 3 features — developmental trajectory per condition",
>    fontsize=13, y=1.02
> 6. Saves as "top_features_over_time.png" (dpi=150) and shows.
> ```

In [ ]:
# Paste the code from your AI here

**Questions to reflect on:**
- In the boxplots, which condition stands out most clearly from the others for each feature?
- In the trajectory plots, do condition differences increase or decrease over time?
- Combining the boxplots and trajectories: would you say the conditions are most different early or late in development?

---
## Section 10 — Summary

> **Prompt:**
> ```
> I have:
> - df: dataframe with columns "DIV" and "Patient_Status"
> - FEATURE_COLS: list of feature names
> - CONDITIONS: list of 4 condition names
> - cv_results: dict from sklearn cross_validate with keys
>   "test_accuracy", "test_balanced_accuracy", "test_f1_macro"
> - perm_df: DataFrame with columns "feature" and "importance_mean"
>
> Write a Python cell that prints a clean summary:
> - Dataset size (n recordings, n features, n conditions, DIV range)
> - Mean +/- std for test accuracy, balanced accuracy, and F1-macro from cv_results
> - Random chance baseline = 1/len(CONDITIONS)
> - Top 3 features by permutation importance with their mean importance scores
> ```

In [ ]:
# Paste the code from your AI here

---
## Final reflection

Answer these questions in your own words (double-click this cell to edit it):

**1. Unsupervised (PCA):** Before showing the model any labels, did the data naturally separate by condition? At which DIV did you first see clear separation between the condition centroids?

*Your answer here...*

**2. Supervised (Random Forest):** How accurately can the model distinguish the four conditions? Is the performance much better than random chance? Is there any sign of overfitting?

*Your answer here...*

**3. Feature importances:** Which three features were most important to the model? What do those features measure biologically, and why might they differ between GBM-patient organoids, astrocyte co-cultures, and controls?

*Your answer here...*

**4. Biological interpretation:** Based on everything you have seen — PCA, classifier performance, feature importances, and developmental trajectories — what is your overall conclusion about how GBM affects the electrophysiology of neural organoids?

*Your answer here...*